In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from pyod.models.lof import LOF
from river.drift import PageHinkley
from hotelling.stats import hotelling_t2
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset
from evidently.metrics import ColumnDriftMetric
from alibi_detect.cd import MMDDrift
from alibi_detect.utils.saving import save_detector, load_detector

In [38]:
df_planta_1 = pd.read_csv('../df_procesados/df_planta_1.csv')
df_planta_1['date_time'] = pd.to_datetime(df_planta_1['date_time'])

In [ ]:
# ========= CONFIG =========
DT_COL = "date_time"
# Variables operativas de entradas (X)
X_COLS = [
    "pH_coagulacion_entrada", "pH_clarificado", "caudal_parshall",
    "caudal_envio_daf", "nivel_ecualizador_1", "nivel_ecualizador_2",
    "temperatura_daf", "conductividad", "turbidez_entrada"
]
# KPI / salida principal
KPI_COL = "turbidez_salida_daf"  # ajusta al nombre real

BASELINE_START = "2025-08-01 00:00:00"
BASELINE_END   = "2025-08-14 23:59:59"

# Ventana móvil (como pediste)
WIN_MIN   = "30min"
STEP_MIN  = "10min"

# Umbrales del “semáforo”
PSI_WARN, PSI_ALARM = 0.10, 0.25
KS_P_ALARM = 0.01

# ==========================

def load_timeseries(path_csv):
    df = pd.read_csv(path_csv)
    df[DT_COL] = pd.to_datetime(df[DT_COL])
    df = df.sort_values(DT_COL).set_index(DT_COL)
    return df

def split_baseline_current(df):
    base = df.loc[BASELINE_START:BASELINE_END]
    cur  = df.loc[BASELINE_END:]
    return base, cur

# ---------- 1) DRIFT UNIVARIADO (EVIDENTLY) ----------
def run_evidently_univariate(base, cur, save_html="drift_univar.html"):
    # Evidently espera un "reference" y un "current"
    rep = Report(metrics=[
        DataDriftPreset(),  # PSI por defecto + tests no paramétricos
        *[ColumnDriftMetric(c) for c in X_COLS + [KPI_COL] if c in base.columns]
    ])
    rep.run(reference_data=base.reset_index(), current_data=cur.reset_index())
    rep.save_html(save_html)
    # También puedes rep.as_dict() si quieres parsear métricas

# ---------- 2) DRIFT MULTIVARIADO (MMD - alibi-detect) ----------
def fit_mmd_baseline(base):
    x_ref = base[X_COLS].dropna().values
    cd = MMDDrift(x_ref, p_val=0.01, kernel='rbf')
    return cd

def test_mmd_window(cd, X_win):
    X = X_win[X_COLS].dropna().values
    if len(X) < 20:
        return {"p_val": 1.0, "is_drift": 0, "distance": 0.0}
    res = cd.predict(X, return_p_val=True, return_distance=True)
    return {"p_val": float(res['data']['p_val']),
            "is_drift": int(res['data']['is_drift']),
            "distance": float(res['data']['distance'])}

# ---------- 3) CONTROL MULTIVARIADO T²/Q ----------
def fit_pca_baseline(base, n_components=None):
    Xb = base[X_COLS].dropna()
    scaler = StandardScaler().fit(Xb)
    Zb = scaler.transform(Xb)
    pca = PCA(n_components=n_components).fit(Zb)
    return scaler, pca

def t2_q_for_window(scaler, pca, X_win):
    Xw = X_win[X_COLS].dropna()
    if Xw.empty:
        return np.nan, np.nan
    Zw = scaler.transform(Xw)
    # T² promedio en la ventana (también puedes reportar máx/p95)
    scores = pca.transform(Zw)
    # T² = z' * Λ^{-1} * z   (usamos varianza de scores)
    var = np.var(scores, axis=0, ddof=1)
    var[var == 0] = 1e-9
    T2_inst = np.sum((scores**2) / var, axis=1)
    T2_mean = float(np.mean(T2_inst))
    # Q (SPE): residuals energy
    Zw_rec = pca.inverse_transform(scores)
    residuals = Zw - Zw_rec
    Q = float(np.mean(np.sum(residuals**2, axis=1)))
    return T2_mean, Q

# ---------- 4) KPI / CONCEPT DRIFT (CUSUM / Page-Hinkley con river) ----------
def page_hinkley_series(s):
    ph = PageHinkley(min_instances=30, delta=0.005, threshold=50.0, alpha=0.9999)
    signals = []
    for v in s.dropna().values:
        ph.update(v)
        signals.append(ph.change_detected)
        if ph.change_detected:
            ph.reset()
    return pd.Series(signals, index=s.dropna().index, dtype=bool)

# ---------- 5) ANOMALÍAS / NUEVOS REGÍMENES ----------
def fit_outlier_models(base):
    Xb = base[X_COLS].dropna().values
    iforest = IsolationForest(contamination=0.02, random_state=0).fit(Xb)
    lof = LOF(contamination=0.02).fit(Xb)
    return iforest, lof

def outlier_ratio(models, X_win):
    X = X_win[X_COLS].dropna().values
    if len(X) == 0:
        return 0.0, 0.0
    iforest, lof = models
    o_if = (iforest.predict(X) == -1).mean()
    o_lof = lof.predict(X).mean()  # 1 = outlier
    return float(o_if), float(o_lof)

# ---------- 6) LOOP EN VENTANAS ----------
def sliding_windows(df, win=WIN_MIN, step=STEP_MIN):
    start = df.index.min()
    end = df.index.max()
    t = start
    while t + pd.Timedelta(win) <= end:
        w = df.loc[t : t + pd.Timedelta(win)]
        yield (t, t + pd.Timedelta(win), w)
        t = t + pd.Timedelta(step)

def run_pipeline(csv_path, html_out_prefix="report"):
    df = load_timeseries(csv_path)
    base, cur = split_baseline_current(df)

    # 1) Evidently univariado
    run_evidently_univariate(base, cur, f"{html_out_prefix}_univar.html")

    # 2) MMD multivariado
    mmd = fit_mmd_baseline(base)

    # 3) PCA + T²/Q
    scaler, pca = fit_pca_baseline(base)

    # 4) KPI drift (Page-Hinkley sobre KPI y también sobre su media EWMA si quieres)
    ph_flags = page_hinkley_series(cur[KPI_COL]) if KPI_COL in cur else pd.Series(dtype=bool)
    ph_flags.to_frame("PH_change").to_csv(f"{html_out_prefix}_kpi_ph.csv")

    # 5) Outliers
    models = fit_outlier_models(base)

    # 6) Consolidación por ventanas
    rows = []
    for t0, t1, w in sliding_windows(cur):
        # MMD (multivariado)
        mmd_res = test_mmd_window(mmd, w)
        # T²/Q
        T2, Q = t2_q_for_window(scaler, pca, w)
        # Outliers
        r_if, r_lof = outlier_ratio(models, w)
        # Semáforo simple (ejemplo): rojo si MMD is_drift y >10% outliers
        level = "Verde"
        if mmd_res["is_drift"] == 1 or r_if > 0.10 or r_lof > 0.10:
            level = "Amarillo"
        if (mmd_res["is_drift"] == 1 and (r_if > 0.10 or r_lof > 0.10)) or (not np.isnan(T2) and T2 > 10):
            level = "Rojo"

        rows.append({
            "t_ini": t0, "t_fin": t1,
            "MMD_p": mmd_res["p_val"], "MMD_is_drift": mmd_res["is_drift"], "MMD_dist": mmd_res["distance"],
            "T2_mean": T2, "Q_mean": Q,
            "outlier_iforest": r_if, "outlier_lof": r_lof,
            "semaforo": level
        })

    pd.DataFrame(rows).to_csv(f"{html_out_prefix}_ventanas.csv", index=False)
    print("Listo: HTML univariado, flags KPI (CSV) y consolidado por ventanas (CSV)")

if __name__ == "__main__":
    # Ejemplo con tu archivo subido:
    run_pipeline("df_planta_1_timeseries.csv", html_out_prefix="planta1")
